# CASA forms: extraction test

Run on a Colab GPU (Runtime > Change runtime type > T4 GPU). Run the cells in order, from the top, in a fresh session.

It clones the code from GitHub (branch `feature/casa-doc01-rules`), uploads your CASA JPG/PNG forms, and writes `doc01_results.json` for download.

In [ ]:
!git clone -b feature/casa-doc01-rules https://github.com/Devadn11206/DOC_Extractor_QUAPT.git /content/repo

In [ ]:
!pip install -q -r /content/repo/requirements.txt rapidocr_onnxruntime accelerate
import transformers
print('transformers', transformers.__version__)
# If the version above is below 5.0, run:  !pip install -q "transformers>=5"
# then Runtime > Restart session and start again from the first cell.

In [ ]:
import sys, types
# Load the CASA modules without the package __init__, which imports unrelated pipeline code.
pkg = types.ModuleType('doc_extractor')
pkg.__path__ = ['/content/repo/src/doc_extractor']
sys.modules['doc_extractor'] = pkg
from doc_extractor.casa_reader import PaddleVLCasaReader, read_casa_form
print('CASA modules loaded')

In [ ]:
from google.colab import files
import os
os.makedirs('/content/doc01', exist_ok=True)
for name, data in files.upload().items():
    open(os.path.join('/content/doc01', name), 'wb').write(data)
print('uploaded to /content/doc01')

In [ ]:
import os, json, glob
from PIL import Image

images = sorted(glob.glob('/content/doc01/*.jpg') + glob.glob('/content/doc01/*.jpeg') + glob.glob('/content/doc01/*.png'))
print(len(images), 'images')
reader = PaddleVLCasaReader()
results = {}
for p in images:
    r = read_casa_form(Image.open(p), reader)
    results[os.path.basename(p)] = r
    print('==', os.path.basename(p), '| status:', r['status'], '| model secs:', r['timing']['model_seconds'])
    for k, v in r['fields'].items():
        print(f"   {k:26s} {str(v['value']):28s} {v['null_reason'] or '':9s} {'REVIEW' if v['review_required'] else 'ok':6s} {v['error'] or ''}")
json.dump(results, open('/content/doc01/doc01_results.json', 'w'), indent=2, ensure_ascii=False, default=str)

In [ ]:
from google.colab import files
files.download('/content/doc01/doc01_results.json')